In [1]:
# Instalamos el paquete de OpenGeoAI, desinstalamos el paquete de torchvision que trae opengeoai e instalamos la version compatible con la GPU P100 de torchvision
%pip install geoai-py -q
%pip uninstall torch torchvision torchaudio -y
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.7/565.7 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 666.8/666.8 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.1/688.1 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.3/859.3 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6

In [2]:
import os
import geoai

In [3]:
input_folder = "/kaggle/input/datasets/davidenriquealba/parcela-b7"
out_folder = "/kaggle/working"

Entrenamos el modelo Mask R-CNN sobre nuestros tiles generados, para que clasifique, localice bboxes y aplique máscaras a cada cepa detectada.

In [4]:
geoai.train_instance_segmentation_model(
    images_dir=f"{input_folder}/images",
    labels_dir=f"{input_folder}/labels",
    output_dir=f"{out_folder}",
    num_classes=2,  # clase fondo y clase cepa. En un futuro, se añadirá clase tronco
    num_channels=5, # 3 para imágenes RGB, 5 para imágenes MSP
    batch_size=4, # Para no consumir excesiva VRAM, y no provocar un error de Out of Memory a mitad de ejecución.
    num_epochs=30, # 10 para una PoC, 50 para entrenamiento real con dataset augmentado.
    learning_rate=0.0005, # Learning rate menos agresivo que el de por defecto, para un descenso de gradiente suave y controlado con un batch sizze de 4. 
    val_split=0.2,
    visualize=True,
    verbose=True,
)

Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:00<00:00, 197MB/s]


Using device: cuda
Found 3230 image files and 3230 label files
Training on 2584 images, validating on 646 images
Epoch: 1, Batch: 1/646, Loss: 4.7021, Time: 3.22s
Epoch: 1, Batch: 11/646, Loss: 3.2992, Time: 12.65s
Epoch: 1, Batch: 21/646, Loss: 3.1365, Time: 12.76s
Epoch: 1, Batch: 31/646, Loss: 2.6786, Time: 12.79s
Epoch: 1, Batch: 41/646, Loss: 2.0869, Time: 12.67s
Epoch: 1, Batch: 51/646, Loss: 1.5266, Time: 12.74s
Epoch: 1, Batch: 61/646, Loss: 1.7136, Time: 12.68s
Epoch: 1, Batch: 71/646, Loss: 1.9537, Time: 12.66s
Epoch: 1, Batch: 81/646, Loss: 1.0360, Time: 12.77s
Epoch: 1, Batch: 91/646, Loss: 2.8515, Time: 12.71s
Epoch: 1, Batch: 101/646, Loss: 1.1031, Time: 12.90s
Epoch: 1, Batch: 111/646, Loss: 1.0910, Time: 12.80s
Epoch: 1, Batch: 121/646, Loss: 1.0180, Time: 12.77s
Epoch: 1, Batch: 131/646, Loss: 1.1444, Time: 12.75s
Epoch: 1, Batch: 141/646, Loss: 1.0112, Time: 12.85s
Epoch: 1, Batch: 151/646, Loss: 0.8435, Time: 12.92s
Epoch: 1, Batch: 161/646, Loss: 0.7826, Time: 12.94

In [5]:
import os
import zipfile
from IPython.display import FileLink

# Ruta donde el modelo ha guardado los pesos (ajusta si la carpeta interna se llama distinto)
ruta_modelos = "/kaggle/working/modelos_tfg"
archivo_zip = "/kaggle/working/mis_pesos_tfg.zip"

# Comprimir la carpeta de modelos
print("Comprimiendo modelos...")
with zipfile.ZipFile(archivo_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(ruta_modelos):
        for file in files:
            file_path = os.path.join(root, file)
            # Guardar en el zip con ruta relativa
            zipf.write(file_path, os.path.relpath(file_path, ruta_modelos))

print("¡Listo para descargar!")
# Generar el enlace interactivo para bajarlo a tu PC
FileLink(r'mis_pesos_tfg.zip')

Comprimiendo modelos...
¡Listo para descargar!


/kaggle/working/mis_pesos_tfg.zip